# 공지 데이터 통합 파이프라인


<!-- hermes:explanation -->
## 통합 파이프라인 개요
이 노트북은 공지 목록 수집, 상세 본문 보강, 이미지 OCR, 마감일 추출, 카테고리 점수화를 한 흐름으로 묶은 통합본입니다. 목적은 전주대학교 공지 데이터를 챗봇 검색용 CSV와 후속 적재 데이터로 정리하는 것입니다.

대표 흐름은 다음과 같습니다.
1. 기존 결과 파일을 불러오거나 빈 데이터프레임을 준비합니다.
2. 공지 목록과 상세 본문을 수집합니다.
3. 이미지 링크와 OCR 텍스트를 보강합니다.
4. 마감일과 카테고리 점수를 계산합니다.
5. 신규 수집분을 기존 데이터와 병합해 결과 파일로 저장합니다.

공개 결과 CSV 기준으로 `04_통합공지_카테고리분류_결과.csv`는 일반공지 498건과 장학공지 446건을 합친 통합공지 결과이고, `05_학사공지_카테고리분류_결과.csv`는 학사공지 482건 결과입니다. 행 수는 헤더를 제외한 기준입니다.


In [27]:
########################################################## 초기 실행 시에만 아래 주석 해제
''' <---- 초기 실행 주석 해제
#step4. OCR 기능을 위한 라이브러리 설치
#-----------------------------------------------
!pip install easyocr
!pip install torch torchvision torchaudio

!apt install -y poppler-utils
!pip install pillow matplotlib
#-----------------------------------------------
''' # <---- 초기 실행 주석 (''') 해제
"""-------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
#공비별 카테고리 키워드
일반_공지 = {
    "학사/수업": [
        "수강", "학기", "학점", "교육과정", "졸업", "교과목", "학사", "성적", "교양", "졸업생", "방학",
        "편입", "전공", "교환학생", "학업", "학위", "학력", "복학", "기숙사 입주",
        "학부", "학과", "수업", "실습", "학사", "수강신청", "교육"],
    "진로/취업": [
        "취업", "진로", "채용", "입사", "직무", "일자리", "인턴", "커리어", "직업", "잡콘서트",
        "일자리", "일자리박람회", "커리어"],
    "장학금/지원금": [
        "장학금", "장학", "장학생", "지원금", "학자금지원", "학자금 지원", "근로장학",  "근로 장학", "성적 장학", "성적장학", "등록금 감면", "장학 재단",
        "장학재단", "학비 지원", "학비지원"],
    "대외활동/서포터즈": [
        "서포터즈", "기자단", "홍보대사", "대외활동", "봉사단", "체험단", "멘토단", "자원봉사",
        "자원봉사자", "활동", "캠프", "행사", "봉사활동", "봉사자", "대사", "봉사", "응원단", "자원봉사", "자치기구"],
    "공모전/경진대회": [
        "공모전", "대회", "경진대회", "콘테스트", "아이디어", "캠프", "경진 대회", "해커톤"],
    "교내행사/이벤트": [
        "행사", "축제", "페스티벌", "박람회", "설명회", "콘서트", "캠프", "포럼", "세미나", "이벤트",
        "연합", "학회", "학술제", "학술대회", "워크숍", "온스타", "onSTAR", "초청", "참가자", "기념",
        "설명", "진행", "공연", "심포지엄", "전시", "라이브", "공모전", "경진대회", "문화제", "공연", "음악회",
        "단체 활동", "대동제", "대회", "페어", "파티", "축하", "컨퍼런스"],
    "비교과/교육": [
        "프로그램", "프로그램 안내", "프로그램안내", "지원 프로그램", "특강", "비교과", "워크숍", "세미나", "교육", "학습법",
        "연수", "강좌", "동아리", "소모임", "영어", "스터디", "멘토링", "온스타", "onSTAR", "예치금","역량",
        "과정", "실습", "학습", "역량 강화", "교육과정", "강의", "자격증", "취득", "학습자", "훈련", "강좌", "SP", "CP",
        "자격", "성장", "훈련", "학습법", "기초", "심화", "학점", "교양", "전공", "학습법", "성적", "공부", "실습", "과제"],
    "군 관련": [
        "예비군", "병역", "군입대", "훈련", "복무", "군대", "입대", "전역", "병사", "장교", "부사관", "훈련소",
        "군사", "복무기간", "병역의무", "군생활", "군 입대", "국방", "훈련소", "기초군사훈련", "동원훈련",
        "병역법", "군인", "군복무", "병역제도", "국방부", "예비군 훈련"],
    "창업 관련": [
          "창업", "창업 캠프"],
    "행정" : ["행정", "결과 안내"],
    "기타" : ["%%%%%%%"]
}

장학_공지 = {
    "국가근로장학금": [
  "국가근로장학금", "하계방학집중근로", "동계방학집중근로", "희망근로지 신청", "추가 신청", "1차 신청", "2차 신청", "3차 신청", "신규장학생 운영", "선발 발표", "사전교육", "근로 기간", "근로 장소", "교내근로", "교외근로", "공공기관", "시급", "최대 근로시간", "업무스케쥴", "출근부", "학업시간표 등록", "오리엔테이션", "부정근로", "근로자격 해제", "중복참여 불가", "장학생 준수사항", "근로지 배정", "선호학과", "근로지 담당자", "출근부 어플", "장학금 지급일", "근로지 대학제출", "학적변동", "휴학", "자퇴", "졸업", "제적", "문의", "시스템 오류", "한국장학재단 홈페이지", "모바일 어플"
],
"교내장학금": [
  "가족장학금", "수퍼스타(동문) 자녀 장학금", "다문화장학금", "장애인자녀장학금", "장애대학생장학금", "특수환경지원장학금", "장학사정관제장학금", "재단우대장학금", "신청기간", "신청자격", "성적기준", "이수학점", "평점평균", "제출서류", "가족관계증명서", "개인정보동의서", "장학금액", "지급방법", "inSTAR", "포털", "신규자", "중복수혜", "수업료 감면", "문의", "학생지원실"
],
"교외장학금": [
  "화성시인재육성재단", "하이트진로홀딩스", "강원랜드 멘토링", "전주시 글로벌 인재양성", "인천인재평생교육진흥원", "서울장학재단", "논산시장학회", "농어촌희망재단", "푸른등대 기부장학금", "부산진구장학회", "대산농촌재단", "고속도로장학재단", "은평구민장학재단", "춘천시민장학재단", "손태희장학재단", "협성문화재단", "롯데장학재단", "우아한 사장님 자녀", "미래산업인재", "지역정착 장학금", "해외연수", "해외봉사", "해외유학", "창업기숙사", "꿈드림", "멘토링", "사회리더", "기부장학", "본인부담 등록금 반값지원", "저소득 대학생", "독립유공자 후손", "북한이탈청소년", "소아청소년 당뇨인", "긴급재난 지원", "멘토단", "멘티", "활동기관", "활동계획서", "시급", "최대근로시간", "활동기간", "제출서류", "재단 홈페이지", "추천서"
],
"국가장학금": [
  "국가장학금", "이공계", "인문100년", "예술체육비전", "주거안정장학금", "신청기간", "1차", "2차", "가구원 동의", "소득분위", "지원구간", "성적기준", "신청방법", "서류제출", "지급", "선발", "학업계획서", "학업시간표", "학적변동", "구제신청", "탈락", "문의", "한국장학재단"
],
"학자금대출": [
  "학자금대출", "대출이자 지원", "신용회복 지원", "특별상환유예", "기등록자 특별승인", "농촌출신 학자금융자", "채무자 신고", "해외이주", "유학신고", "업무처리기준", "신청기간", "신청방법", "심사", "지급", "문의", "상환", "학자금지원사업", "결혼이민자", "혼인귀화자", "대학학비지원", "지방자치단체", "농어촌", "이자지원사업", "상환유예", "신용회복"
],
"멘토링/청소년교육지원": [
  "대학생 청소년교육지원사업", "멘토링", "사회리더", "멘토", "멘티", "활동계획서", "활동기관", "사전신청", "사전교육", "활동기간", "활동내용", "시급", "최대근로시간", "중복참여", "부정근로", "선발", "결과발표", "지급", "지급일", "지급방법", "온라인교육", "매뉴얼", "시스템", "어플", "홈페이지", "활동 포기", "기관-학생 상호평가", "출근부", "학업시간표 입력", "업무스케쥴", "활동기관 발굴", "신규기관 등록", "중복참여불가", "이해관계 회피", "활동시간 제한"
],
"특별지원/목회자/북한이탈/보훈": [
  "목회자 장학금", "목회자 자녀 장학금", "북한이탈주민 교육지원금", "보훈장학금", "국가보훈대상자", "국가유공자", "교육지원대상자증명서", "대학수업료등면제대상자증명서", "가족관계증명서", "재직증명서", "교회주보", "결혼이민자", "혼인귀화자", "다문화", "장애인자녀", "장애대학생", "특수환경지원", "아동복지시설", "한부모가정", "긴급재난 지원", "제출서류", "신청기간", "지급방법"
],
"글로벌/해외연수/파란사다리": [
  "글로벌 인재양성", "영어능력강화", "해외연수", "어학연수", "교환학생", "파란사다리", "Fulbright", "국제교류", "신청기간", "선발", "지원", "서류제출", "학점인정", "학적변동", "재단 홈페이지"
],
"취업연계/창업/고졸후학습자/희망사다리": [
  "중소기업 취업연계 장학금", "희망사다리", "고졸후학습자", "창업", "취업", "취업지원", "창업지원", "직무기초교육", "의무종사", "취업연계", "창업지원금", "의무재직", "일자리사업 중복참여", "사용계획서", "진단평가", "보증보험", "취업연계 상담센터", "지원사업", "심사기준", "지원금", "등록금 전액지원", "의무종사 기업기준", "재직확인"
],
"행정/안내/기타": [
  "통학로", "학생지원", "정보로", "안내", "제한대학", "지원대상", "지원방법", "제출서류", "문의", "본인부담", "등록금", "반값지원", "지역정착", "졸업", "휴학", "자퇴", "제적", "졸업예정자", "졸업유예", "학점", "평점", "성적", "학업", "학적변동", "inSTAR", "포털", "이메일", "방문접수", "팩스", "제출기한", "신청기한", "지급기한", "시스템오류", "어플오류", "홈페이지오류", "콜센터", "공지사항", "개인정보동의", "계좌등록", "지급계좌", "통장", "담당자", "연락처"
]
}
학사_공지 = {
    "수업/강의 관련": ["강의", "수업", "교과목", "강의실", "강좌", "시간표", "수강신청", "보강"],
    "시험/평가": ["시험", "수시고사", "중간고사", "기말고사", "평가", "성적", "과제", "응시"],
    "졸업/학위": ["졸업", "논문", "학위", "심사", "학점", "수료", "취업", "조기"],
    "등록/휴학/복학": ["등록", "휴학", "복학", "신청", "복학생", "편입", "자퇴"],
    "장학/재정": ["장학금", "지원", "신청서", "선발", "지급", "장학생"],
    "행사/일정": ["설명회", "일정", "행사", "참석", "기념일", "주간", "날짜", "개강", "종강", "연휴"],
    "행정/공지": ["공지", "안내", "서류", "제출", "처리", "관리", "참조", "변경"]
}
부서_분류 = {
    "학사/수업": ["학생 지원", "학생지원", "학생 역량", "학생역량", "학사지원", "학사 지원", "LINC"],
    "진로/취업":  ["대학 일자리", "대학일자리", "진로취업", "진로 취업", "학생 성공", "학생성공"],
    "장학금/지원금": [],
    "대외활동/서포터즈": [],
    "공모전/경진대회":[],
    "교내행사/이벤트": ["대학 일자리", "대학일자리", "진로취업", "진로 취업" "학생 성공", "학생성공"],
    "비교과/교육": ["대학 일자리", "대학일자리", "진로취업", "진로 취업" "학생 성공", "학생성공"],
    "군 관련": ["예비군"],
}
"""------------------------------------------------------------------------------------------------------------------------------------------------"""

from concurrent.futures import ThreadPoolExecutor
import easyocr
import matplotlib.pyplot as plt
import numpy as np
import cv2
from PIL import Image
from io import BytesIO
from dateutil.parser import parse
from typing import Optional
import calendar
import requests
import pandas as pd
import os
import re
from bs4 import BeautifulSoup as bs
from urllib.parse import urljoin
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

"""-----------------------------------------------------------------------------------------------------------------------------------------------------------------------"""

################################################################## 기존 데이터 관리 함수
def load_or_init_data(filename='04_통합공지_카테고리분류_결과.csv', required_columns  = [
        '공지분류', '등록 번호', '기본 제목', '공지 제목',
        '부서', '등록일', '조회수', '링크', '본문 내용',
        '이미지 링크', '이미지 텍스트', '마감일자', '모집 현황'
    ]):

    if os.path.exists(filename):
        # 1. 기존 파일 불러오기 (필요한 열만 선택)
        df = pd.read_csv(filename, dtype={'등록 번호': str})
        df['등록 번호'] = df['등록 번호'].astype(str)

        # 2. 필요한 열만 남기기 (없는 열은 빈 값으로 생성)
        for col in required_columns:
            if col not in df.columns:
                if col == '조회수':
                    df[col] = 0  # 조회수는 0으로
                elif col in ['등록일', '마감일자']:
                    df[col] = pd.NaT  # 날짜형은 NaT
                else:
                    df[col] = ''  # 문자열은 빈 값

        # 3. 컬럼 순서 보장 및 저장
        df = df[required_columns]
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        return df

    else:
        # 4. 파일이 없으면 빈 DataFrame 생성
        return pd.DataFrame(columns=required_columns)

################################################################## 기존 데이터 -> 상단 공지 확인
def get_max_num(temp_df, notice_type):
    filtered = temp_df[
        (temp_df['공지분류'] == notice_type) &
        (temp_df['등록 번호'] != '공지')
    ].copy()

    if filtered.empty:
        return 0

    # 강화된 유효성 검증
    filtered['등록 번호'] = filtered['등록 번호'].astype(str).str.strip()
    valid_mask = (
        filtered['등록 번호'].str.match(r'^\d+$') &  # 순수 숫자 형식 검증
        filtered['등록 번호'].ne('0')  # 0 값 제외
    )
    filtered = filtered[valid_mask]

    nums = pd.to_numeric(filtered['등록 번호'], errors='coerce')
    valid_nums = nums.dropna()

    return int(valid_nums.max()) if not valid_nums.empty else 0





"""--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
############################################################ ## step1. 기본 크롤링
def crawl_step1(Noti_title, temp_df, T_limit=500, T_max_pages=1):
    if Noti_title == '일반공지':
        NOTI = '01'
    elif Noti_title == '학사공지':
        NOTI = '02'
    elif Noti_title == '장학공지':
        NOTI = '03'
    else:
        raise ValueError("공지 유형은 '일반공지', '학사공지', '장학공지' 중 하나여야 합니다.")

    def crawl_page(limit, offset, end=0):
        url = f"https://www.jj.ac.kr/jj/community/notice{NOTI}.do?mode=list&&articleLimit={limit}&article.offset={offset}"
        try:
            res = requests.get(url, timeout=5)
            res.encoding = 'utf-8'
            res.raise_for_status()
        except Exception as e:
            print(f"[!] 요청 실패 (offset={offset}) → {e}")
            return []

        soup = bs(res.text, "html.parser")
        box = soup.find("tbody")
        rows = []

        if not box:
            return rows

        for tr in box.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) < 2:
                continue

            no = tds[0].text.strip()
            if no == str(end):
              break
            content_td = tds[1]

            a_tag = content_td.select_one("div > div > a")
            title = ' '.join(a_tag.stripped_strings) if a_tag else None
            href = a_tag.get("href") if a_tag else None
            link = urljoin(f"https://www.jj.ac.kr/jj/community/notice{NOTI}.do", href) if href else None
            dept_span = content_td.select_one("div > div:nth-of-type(3) > span.b-writer")
            dept = dept_span.text.strip() if dept_span else None
            date_span = content_td.select_one("div > div:nth-of-type(3) > span.b-date")
            date = date_span.text.strip() if date_span else None
            hit_span = content_td.select_one("div > div:nth-of-type(3) > span.b-hit")
            views = int(re.search(r'\d+', hit_span.text.strip()).group()) if hit_span else 0

            if no == "공지":
                if not dept:
                    dept = "관리자"
                date = datetime.today().strftime('%Y-%m-%d')

            prim_key = f"{dept}+{title}+{date}" if title else None

            if title:
                rows.append({
                    "공지분류": Noti_title,
                    "등록 번호": no,
                    "기본 제목": prim_key,
                    "공지 제목": title,
                    "부서": dept,
                    "등록일": date,
                    "조회수": views,
                    "링크": link
                })

        return rows
    """ 크롤링1 통합 함수 """
    def gongji_crawl_parallel(limit=100, max_pages=80):
        offsets = [page * limit for page in range(max_pages)]
        all_rows = []
        end = get_max_num(temp_df, Noti_title)

        with ThreadPoolExecutor(max_workers=10) as executor:
            results = list(executor.map(lambda offset: crawl_page(limit, offset, end), offsets))

        for page_rows in results:
            all_rows.extend(page_rows)

        df = pd.DataFrame(all_rows)
        df.drop_duplicates(subset=["공지 제목", "링크"], inplace=True)
        return df

    try:
        Tstep1_df = gongji_crawl_parallel(T_limit, T_max_pages)
        print(f"  ▷ {Noti_title} 완료: {len(Tstep1_df)}건")
    except Exception as e:
        print(f"  ▷[x] {Noti_title} 실패")
        print(f"에러 내용: {e}")
        Tstep1_df = pd.DataFrame()

    return Tstep1_df

"""--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
########################### step2. 본문 크롤링 + 공지사항 추가 정보 추출 (최종버전)
def extract_notice_body(url):
    try:
        res = requests.get(url, timeout=5)
        res.encoding = 'utf-8'
        soup = bs(res.text, "html.parser")

        # 1. 본문 내용 추출
        content_div = soup.select_one("div.b-content-box div.fr-view")
        content = content_div.get_text(separator="", strip=True) if content_div else None

        # 2. 메타 정보 초기화
        meta_info = {'dept': None, 'date': None, 'content': content}

        # 3. 정보 추출 로직 (강화된 버전)
        date_span = soup.select_one("li.b-date-box span:nth-of-type(2)")
        dept_span = soup.select_one("li.b-writer-box span:nth-of-type(2)")

        if date_span:
            meta_info['date'] = date_span.get_text(strip=True)
        if dept_span:
            meta_info['dept'] = dept_span.get_text(strip=True)

        return meta_info

    except Exception as e:
        print(f"[오류 발생] {url} → {e}")
        return {'content': None, 'dept': None, 'date': None}

def add_notice_info_to_df(df):
    """크롤링한 정보를 DataFrame에 입력"""
    temp_df = df.copy()
    contents = []
    depts = []
    dates = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="  ▷ 본문 크롤링 진행"):
        url = row["링크"]
        result = extract_notice_body(url)

        contents.append(result['content'])
        depts.append(result['dept'])
        dates.append(result['date'])

        # 4. 공지사항인 경우 기본 제목 생성
        if row["등록 번호"] == "공지":
            dept = result['dept'] or "관리자"
            date = result['date'] or datetime.today().strftime('%Y-%m-%d')
            title = row["공지 제목"]
            temp_df.at[idx, "기본 제목"] = f"{dept}+{title}+{date}"

    temp_df["본문 내용"] = contents
    temp_df["부서"] = depts
    temp_df["등록일"] = dates

    return temp_df

"""------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
#공지사항에 있는 이미지 파일 링크 추출
#33333333333333333333333333333333333333333333333333333333333333333 🔹 이미지 링크 추출 함수
def extract_image_links(url):
    try:
        res = requests.get(url, timeout=5)
        res.encoding = 'utf-8'
        soup = bs(res.text, "html.parser")
        img_tags = soup.select("div.b-content-box div.fr-view img")

        # 🔹 이미지 링크 수집
        img_urls = []
        for img in img_tags:
            src = img.get("src")
            if src:
                # 절대경로가 아니면 urljoin으로 붙이기
                full_url = urljoin("https://www.jj.ac.kr", src)
                img_urls.append(full_url)

        # 🔹 리스트가 비어 있으면 None 반환
        if img_urls:
            # 🔸 문자열 형태로 반환 (리스트 [] 제거)
            return ", ".join(img_urls)
        else:
            return None
    except Exception as e:
        print(f"[이미지 크롤링 오류] {url} → {e}")
        return None
###############################################3333333333333333333333333333333333333333333333
def get_existing_notice_titles(existing_df: pd.DataFrame) -> list:
    """기존 데이터에서 '공지' 항목의 기본 제목 추출"""
    return existing_df[
        existing_df['등록 번호'] == '공지'
    ]['기본 제목'].tolist()


#333333333333333333333333333333333333333333333333333333333333 🔹 데이터프레임 병합 함수
def merge_image_links(new_df: pd.DataFrame, existing_df: pd.DataFrame) -> pd.DataFrame:
    # 기존 공지 제목 리스트 추출
    existing_titles = get_existing_notice_titles(existing_df)

    # 기존 데이터에서 이미지 링크 매핑 테이블 생성
    title_to_image = existing_df.set_index('기본 제목')['이미지 링크'].to_dict()
    title_to_link = existing_df.set_index('기본 제목')['링크'].to_dict()
    title_to_date = existing_df.set_index('기본 제목')['등록일'].to_dict()

    # 신규 데이터에 이미지 링크 병합
    for idx, row in tqdm(new_df.iterrows(), total=len(new_df)):
        title = row['기본 제목']

        # 기존 데이터에 존재하는 경우
        if title in existing_titles:
            # 기존 정보 재사용
            new_df.at[idx, '이미지 링크'] = title_to_image.get(title)
            new_df.at[idx, '링크'] = title_to_link.get(title)  # 링크 갱신 방지
            new_df.at[idx, '등록일'] = title_to_date.get(title)  # 등록일 보존
            continue

        # 신규 항목인 경우 크롤링 수행
        url = row['링크']
        img_links = extract_image_links(url)
        new_df.at[idx, '이미지 링크'] = img_links

    return new_df


"""---------------------------------------------------------------------------------------------------------------------------------------------------------------------"""

# Step4. 이미지 파일의 링크를 읽어와서 텍스트 인식 수행
# 시간이 매우 많이 소요되므로 서버에서 실행 권장

# 따로 실행 시킬때는 아래 주석 제거 후 step4의 전체 코드 복사해서 실행

# 🔹 OCR 모델 초기화
reader = easyocr.Reader(['ko', 'en'])

# 🔹 이미지 링크에서 OCR 수행 함수
def ocr_image_links(img_urls):
    if not img_urls or pd.isna(img_urls):
        return None  # 이미지가 없는 경우

    # 이미지 링크가 여러 개일 수 있으므로 리스트로 변환
    img_list = img_urls.split(", ") if isinstance(img_urls, str) else [img_urls]

    # 🔹 각 이미지에 대해 OCR 수행
    texts = []
    for img_url in img_list:
        try:
            # 이미지 다운로드
            response = requests.get(img_url.strip())
            img = Image.open(BytesIO(response.content))
            img_np = np.array(img)

            # 🔹 OCR 수행
            result = reader.readtext(img_np, detail=0)
            texts.append(" ".join(result))
        except Exception as e:
            print(f"[이미지 OCR 오류] {img_url} → {e}")

    # 🔹 여러 이미지의 텍스트를 합쳐서 반환
    return " ".join(texts) if texts else None

# 🔹 데이터프레임 병합 함수
def df_to_ocr_image_links(df, existing_df=None):
    """중복 OCR 처리 방지 기능이 추가된 함수"""

    # 기존 데이터 매핑 테이블 생성
    existing_text_map = {}
    if existing_df is not None:
        existing_text_map = existing_df.set_index('기본 제목')['이미지 텍스트'].to_dict()

    # OCR 처리 진행
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        title = row['기본 제목']

        # 1. 기존 데이터 재사용 조건
        if title in existing_text_map:
            # CASE 1: 기존 텍스트가 유효한 경우
            if pd.notna(existing_text_map[title]):
                df.at[idx, "이미지 텍스트"] = existing_text_map[title]
                continue

            # CASE 2: 기존에 OCR 실패한 경우 재시도
            img_urls = row.get("이미지 링크", "")
        else:
            # CASE 3: 완전히 새로운 항목
            img_urls = row.get("이미지 링크", "")

        # 2. 실제 OCR 수행
        img_texts = ocr_image_links(img_urls)
        df.at[idx, "이미지 텍스트"] = img_texts

    return df


"""-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"""

########################################## step5. (현재 수정 중인 코드) -- 공지 내용에서 마감일 추출하여,[마감일자] 추가

def preprocess_text(text: str) -> str:
    """지능형 전처리: 핵심 정보 보존, 노이즈 제거"""
    # 이메일/전화번호 제거 (날짜 패턴 보호)
    text = re.sub(r'\S+@\S+', '', text)  # 이메일
    text = re.sub(r'\d{2,4}-\d{3,4}-\d{4}', '', text)  # 전화번호
    # IR52 같은 코드명은 날짜 형식이 아니면 제거 (숫자-문자 조합, 단일 코드명)
    text = re.sub(r'\b(?!\d+[./]\d+)[A-Za-z]+\d+\b', '', text)
    # 요일 괄호 제거 (수), (금)
    text = re.sub(r'\([^)]*\)', '', text)
    # 날짜 형식 정규화 (2025. 4. 4 → 2025.4.4)
    text = re.sub(r'(\d+)\.\s+(\d+)', r'\1.\2', text)
    # 공백 정규화
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_until_deadline(text: str, base_year: int) -> Optional[str]:
    def build_date(year, month, day):
        try:
            if year < 100: year += 2000  # 2자리 연도 처리
            if 2000 <= year <= 2100 and 1 <= month <= 12 and 1 <= day <= 31:
                return f"{year}-{month:02d}-{day:02d}"
        except: pass
        return None

    text = re.sub(r'\([^)]*\)', '', text)  # 요일 제거

    # 패턴 그룹: (정규식, 그룹 인덱스, 후처리함수)
    patterns = [
        # 1. ~까지 신청/접수/마감
        (r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})일?\s*까지\s*(신청|접수|마감)', lambda m: build_date(int(m[1]), int(m[2]), int(m[3]))),

        # 2. YYYY년 MM월 DD일 ~ DD일 학생 모집
        (r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})[일\s]*[~\-]+\s*(\d{1,2})[일]?[^\n\r가-힣0-9]*.*?학생\s*모집', lambda m: build_date(int(m[1]), int(m[2]), int(m[4]))),

        # 3. 모집/신청/접수 기간
        (r'(서류\s*접수|접수\s*기간|신청\s*기간|모집\s*기간|모집기간|접수기간|신청기간|지원서\s*접수)[\s:：-]*'
         r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})일?\s*~\s*'
         r'(\d{4})?[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})일?',
         lambda m: build_date(int(m[5]) if m[5] else int(m[2]), int(m[6]), int(m[7]))),

        # 4. 마감 키워드 + 날짜
        (r'(마감|접수종료|신청마감)[\s:：]+'
         r'(\d{4})?[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})일?',
         lambda m: build_date(int(m[2]) if m[2] else base_year, int(m[3]), int(m[4]))),

        # 5. 상시모집 ~ 연도.월
        (r'상시모집.*~\s*(\d{4})[./년\s]*(\d{1,2})[월]?',
         lambda m: build_date(int(m[1]), int(m[2]), calendar.monthrange(int(m[1]), int(m[2]))[1])),

        # 6-1: YYYY.MM.DD HH:MM ~ MM.DD HH:MM
        (r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2}).*?~\s*'
         r'(\d{1,2})[./월\s]*(\d{1,2})',
         lambda m: build_date(int(m[1]), int(m[4]), int(m[5]))),

        # 6-2: ~YYYY.MM.DD HH:MM까지
        (r'~\s*(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2}).*?(?:까지|시)',
         lambda m: build_date(int(m[1]), int(m[2]), int(m[3]))),

        # 6-3: YYYY.MM.DD HH:MM ~ YYYY.MM.DD HH:MM
        (r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2}).*?~\s*'
         r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})',
         lambda m: build_date(int(m[4]), int(m[5]), int(m[6]))),

        # 6-4: 2자리 연도 ~ 2자리 연도
        (r'(\d{2})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})\s*[~\-]\s*'
         r'(\d{2})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})',
         lambda m: build_date(2000 + int(m[4]), int(m[5]), int(m[6])))
    ]

    for pattern, extractor in patterns:
        if (match := re.search(pattern, text, re.IGNORECASE)):
            result = extractor(match)
            if result: return result

    return None


def enhanced_extractor(text: str, base_year: int) -> str:
    """최종 추출 엔진 (우선순위 개선)"""
    cleaned_text = preprocess_text(text)


    if deadline := extract_until_deadline(cleaned_text, base_year):
        return deadline


    # 안전하게 '-' 반환
    return '-'


def extract_deadlines(df: pd.DataFrame) -> pd.DataFrame:
    """마감일자 추출 전용 함수"""
    def get_base_year(reg_date: str) -> int:
        try:
            return parse(reg_date, fuzzy=True).year
        except:
            return datetime.now().year

    def process_row(row):
        base_year = get_base_year(str(row.get('등록일', '')))

        # 1. 본문 내용 우선 처리
        main_text = str(row.get('본문 내용', ''))
        deadline = enhanced_extractor(main_text, base_year)
        if deadline != '-':
            return deadline

        # 2. 이미지 텍스트 차선 처리
        image_text = str(row.get('이미지 텍스트', ''))
        deadline = enhanced_extractor(image_text, base_year)
        if deadline != '-':
            return deadline

        # 3. 전체 텍스트 폴백 처리
        combined = ' '.join([
            str(row.get('공지 제목', '')),
            main_text,
            image_text
        ])
        deadline = enhanced_extractor(combined, base_year)
        return deadline

    df['마감일자'] = df.apply(process_row, axis=1)
    df['모집 현황'] = None
    return df
"""--------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
############################################################## step6. 필수 컬럼 리스트 (순서 보장)
required_columns = [
    '공지분류', '등록 번호', '기본 제목', '공지 제목',
    '부서', '등록일', '조회수', '링크', '본문 내용',
    '이미지 링크', '이미지 텍스트', '마감일자', '모집 현황'
]

# 누락된 컬럼 추가
def index_test(df: pd.DataFrame) -> pd.DataFrame:
  for col in required_columns:
      if col not in df.columns:
          # 데이터 타입에 맞게 초기화
          if col == '조회수':
              df[col] = 0  # 숫자형
          elif col in ['등록일', '마감일자']:
              df[col] = pd.NaT  # 날짜형
          else:
              df[col] = ''  # 문자열
  return df

"""--------------------------------------------------------------------------------------------------------------------------------------------------------------------"""

####################################################============================= step7 데이터 병합 함수
def merge_notice_data(new_df: pd.DataFrame, existing_df: pd.DataFrame) -> pd.DataFrame:
    merged_total_df = pd.DataFrame()
    category_list = new_df["공지분류"].unique()

    for category in category_list:
        df6 = new_df[new_df["공지분류"] == category].copy()
        ex_df = existing_df[existing_df["공지분류"] == category].copy()

        # Step 1: 공지 항목 비교 및 Dump 처리
        df6_titles = set(df6[df6["등록 번호"] == "공지"]["기본 제목"])
        ex_df["등록 번호"] = ex_df.apply(
            lambda row: 0 if row["등록 번호"] == "공지" and row["기본 제목"] not in df6_titles else row["등록 번호"],
            axis=1,
        )

        # Step 2: 공지 항목 통합
        noti_df = pd.concat([
            ex_df[ex_df["등록 번호"] == "공지"],
            df6[df6["등록 번호"] == "공지"]
        ]).drop_duplicates(subset=["기본 제목"], keep="first")

        # Step 3: 일반 항목 병합
        df6_normal = df6[df6["등록 번호"] != "공지"]
        ex_df_normal = ex_df[ex_df["등록 번호"] != "공지"]

        # 중복 제거
        duplicate_titles = set(df6_normal["기본 제목"]) & set(ex_df_normal["기본 제목"])
        df6_normal = df6_normal[~df6_normal["기본 제목"].isin(duplicate_titles)]

        combined_normal = pd.concat([df6_normal, ex_df_normal])
        combined_normal = combined_normal.drop_duplicates(subset=["기본 제목"], keep="first")

        # 등록번호 재계산 (핵심 수정 부분)
        # --------------------------------------------
        # 1. 기존+신규 데이터 통합 분석
        combined_numbers = pd.concat([
            ex_df_normal["등록 번호"],
            df6_normal["등록 번호"]
        ]).astype(str).str.strip()

        # 2. 유효한 숫자 필터링
        numeric_mask = combined_numbers.str.match(r'^\d+$')  # 순수 숫자만 허용
        valid_numbers = pd.to_numeric(
            combined_numbers[numeric_mask],
            errors='coerce'
        ).dropna().astype(int)

        # 3. 최대값 계산
        top_number = valid_numbers.max() if not valid_numbers.empty else 0
        if top_number == 0:  # 데이터가 전혀 없는 경우
            top_number = 1  # 1부터 시작

        # 4. 번호 부여
        combined_normal["등록일"] = pd.to_datetime(combined_normal["등록일"], errors="coerce")
        combined_normal = combined_normal.sort_values("등록일", ascending=False)
        new_numbers = list(range(top_number, top_number - len(combined_normal), -1))
        combined_normal["등록 번호"] = new_numbers
        # --------------------------------------------

        # 형식 맞추기
        combined_normal["등록일"] = combined_normal["등록일"].dt.strftime("%y-%m-%d")

        # 최종 병합
        final_df = pd.concat([noti_df, combined_normal])
        merged_total_df = pd.concat([merged_total_df, final_df])

    return merged_total_df.reset_index(drop=True)


"""-------------------------------------------------------------------------------------------------------------------------------------------------------------------"""

#################################################################################-- step8 모집 현황 처리 함수
def determine_recruitment_status(df: pd.DataFrame) -> pd.DataFrame:
    """모집현황 판단 전용 함수 (마감일자 컬럼 필요)"""
    def check_priority(row):
        combined_text = ' '.join([
            str(row.get('공지 제목', '')),
            str(row.get('본문 내용', '')),
            str(row.get('이미지 텍스트', ''))
        ])
        deadline = str(row.get('마감일자', '')).strip()

        # 1. 마감일자가 없고 선착순 키워드 포함 시
        if deadline == '-' and '선착순' in combined_text:
            return '선착순'

        # 2. 마감일 기준 모집중/마감 판단
        try:
            if deadline and deadline != '-':
                deadline_date = datetime.strptime(deadline, "%Y-%m-%d").date()
                today = datetime.today().date()
                return '모집중' if deadline_date >= today else '마감'
            return '정보 없음'
        except:
            return '정보 없음'

    df['모집 현황'] = df.apply(check_priority, axis=1)
    return df

"""-------------------------------------------------------------------------------------------------------------------------------------------------------------------"""

# 🔹 키워드 정의 (간략 버전)
category_dict = {
    "일반공지": 일반_공지,
    "장학공지": 장학_공지,
    "학사공지": 학사_공지
}
department_dict = {
    "일반공지": 부서_분류,
    "장학공지": 부서_분류,
    "학사공지": 부서_분류
}
# 가중치 설정
TITLE_WEIGHT = 3
CONTENT_WEIGHT = 2
DEPT_WEIGHT = 0
IMAGE_WEIGHT = 2

def calculate_category_scores_by_notice_type(df: pd.DataFrame) -> pd.DataFrame:

    def calculate_score(title, content, dept, image, category_keywords, department_keywords):
        title = str(title) if pd.notna(title) else ""
        content = str(content) if pd.notna(content) else ""
        dept = str(dept) if pd.notna(dept) else ""
        image = str(image) if pd.notna(image) else ""

        scores = {k: 0 for k in category_keywords.keys()}

        for cat, keywords in category_keywords.items():
            scores[cat] += (sum(title.count(k) for k in keywords) * TITLE_WEIGHT * 10) / (len(title) + 1)
            scores[cat] += (sum(content.count(k) for k in keywords) * CONTENT_WEIGHT * 10) / (len(content) + 1)
            scores[cat] += (sum(image.count(k) for k in keywords) * IMAGE_WEIGHT * 10) / (len(image) + 1)

        for cat, keywords in department_keywords.items():
            if cat in scores:
                scores[cat] += (sum(dept.count(k) for k in keywords) * DEPT_WEIGHT * 10) / (len(dept) + 1)

        if all(v == 0 for v in scores.values()):
            scores["기타"] = 1.0

        return scores

    # 점수 계산 및 열 추가
    result_df = df.copy()
    for idx, row in result_df.iterrows():
        noti_type = row.get("공지분류", "").strip()
        category_keywords = category_dict.get(noti_type, {})
        department_keywords = department_dict.get(noti_type, {})

        title = row.get("공지 제목", "")
        content = row.get("본문 내용", "")
        dept = row.get("부서", "")
        image = row.get("이미지 텍스트", "")

        scores = calculate_score(title, content, dept, image, category_keywords, department_keywords)

        for cat, score in scores.items():
            colname = f"{noti_type}_{cat}"
            if colname not in result_df.columns:
                result_df[colname] = 0.0
            result_df.at[idx, colname] = score

    return result_df

"""------------------------------------------------------------------------------------------------------------------------------------------------"""
#=================================== 00 데이터 로드 ===================================# <----- 데이터 로드
existing_df = load_or_init_data('04_통합공지_카테고리분류_결과.csv')                                                                                           # <--------- 초기 데이터와, 저장 데이터 이름 / 저장 위치가 동일해야함
existing_df2 = load_or_init_data('05_학사공지_카테고리분류_결과.csv',
        ['공지분류', '등록 번호', '기본 제목', '공지 제목', '부서', '등록일', '조회수', '링크', '본문 내용'])                                                    # <--------- 초기 데이터와, 저장 데이터 이름 / 저장 위치가 동일해야함
#======================================================== step1 실행 함수 =============# <----- 실제 실행시에는 각 head()안에 숫자를 1에서 500으로 변경

print("\n■ step1 시작")
df_general = crawl_step1("일반공지", existing_df)
df_scholarship = crawl_step1("장학공지", existing_df)
step1_df = pd.concat([df_general, df_scholarship], ignore_index=True)
HS_step1_df = crawl_step1("학사공지", existing_df2)
#======================================================================================#
#=================================== step 2 실행 함수 =================================#

#공지사항의 [본문내용 카테고리] 열 추가 ---> 시간 약간 소요 됨
print("\n■ step2 시작")
try:
    # 일반+장학
    step2_df = add_notice_info_to_df(step1_df)
    step2_df.to_csv(f"step2_①통합공지(본문내용).csv", index=False, encoding='utf-8-sig')
    print(f"     ㄴ 완료 : 통합공지_(본문내용)")
    # 학사
    HS_step2_df = add_notice_info_to_df(HS_step1_df)
    HS_step2_df.to_csv(f"step2_②학사공지(본문내용).csv", index=False, encoding='utf-8-sig')
    print(f"\n     ㄴ 완료 : 학사공지_(본문내용)")
except Exception as e:
    print("\n    <x> step2 실패")
    print(f"에러 내용: {e}")

#======================================================================================#
#======================================== step3 실행 함수 =============================#
# 공지사항에 있는 이미지 파일 링크를 추출하여 "이미지 링크"열 추가
# 약 10분 소요
print("■ step3 시작")
try:
    step3_df = merge_image_links(step2_df, existing_df)
    # 파일 저장
    step3_df.to_csv(f"step3_①통합공지(이미지링크).csv", index=False, encoding='utf-8-sig')
    print(f"  ▷ 완료 :통합공지(이미지링크)")
except Exception as e:
    print("  ▷ 실패")
    print(f"에러 내용: {e}")
#======================================================================================#
#============================= Step4 실행 함수 ========================================#

#이미지 파일의 링크를 읽어와서 텍스트 인식 수행
print("■ step4 시작")
try:
    # 🔹 데이터프레임에 이미지 텍스트 병합 (시간 소요 가능)
    step4_df = df_to_ocr_image_links(step3_df, existing_df)
    step4_df.to_csv(f"step4_①통합공지(이미지텍스트).csv", index=False, encoding='utf-8-sig')
    print(f"  ▷ 완료 : 통합공지(이미지텍스트)")
except Exception as e:
    print("  ▷ 실패")
    print(f"에러 내용: {e}")
#=====================================================================================#
#============================================== step5 실행 함수 ======================#

# 시작) 공지 내용에서 마감일 추출하여,[마감일자] 추가
print("■ step5 시작")
try:
    # 🔹 마감일 추출 및 열 추가 (예: "모집 마감일" 등)
    step5_df = extract_deadlines(step4_df)
    step5_df.to_csv(f"step5_①통합공지(마감일).csv", index=False, encoding='utf-8-sig')
    print(f"  ▷ 완료 : 통합공지(마감일)")

except Exception as e:
    print("  ▷ 실패")
    print(f"에러 내용: {e}")
#====================================================================================#
#================================================== step6 실행 함수 =================#

# 컬럼 순서 재정렬
print("■ step6 시작")
try:
    step6_df = index_test(step5_df)
    step6_df = step6_df.reindex(columns=required_columns)
    step6_df.to_csv(f"step6_①통합공지(컬럼정렬).csv", index=False, encoding='utf-8-sig')
    print(f"  ▷ 완료 : 통합공지(컬럼정렬)")

except Exception as e:
    print("  ▷ 실패")
    print(f"에러 내용: {e}")
#====================================================================================#
#================================================== step7 실행 함수 =================#

# 기존 데이터와 병합
print("■ step7 시작")
try:
    step7_df = merge_notice_data(step6_df, existing_df)
    step7_df.to_csv(f"step7_①통합공지(데이터병합).csv", index=False, encoding='utf-8-sig')
    print(f"  ▷ 완료 : 통합공지(데이터병합)")

    HS_step7_df = merge_notice_data(HS_step2_df, existing_df2)
    HS_step7_df.to_csv(f"step7_②학사공지(데이터병합).csv", index=False, encoding='utf-8-sig')
    print(f"  ▷ 완료 : 학사공지(데이터병합)")
except Exception as e:
    print("  ▷ 실패")
    print(f"에러 내용: {e}")
#====================================================================================#
#================================================== step8 실행 함수 =================#

# 기존 데이터와 병합
print("■ step8 시작")
try:
    step8_df = determine_recruitment_status(step7_df)
    step8_df.to_csv(f"step8_①통합공지(마감현황).csv", index=False, encoding='utf-8-sig')
    print(f"  ▷ 완료 : 통합공지(마감현황)")

except Exception as e:
    print("  ▷ 실패")
    print(f"에러 내용: {e}")
#====================================================================================#
#================================================== final 실행 함수 =================#

# 기존 데이터와 병합
print("■ final 시작")
try:
    step9_df = calculate_category_scores_by_notice_type(step8_df)
    step9_df.to_csv(f"04_통합공지_카테고리분류_결과.csv", index=False, encoding='utf-8-sig') # <--------- 초기 데이터와, 저장 데이터 이름 / 저장 위치가 동일해야함
    print(f"  ▷ 완료 : 통합공지(카테고리)")

    HS_step9_df =calculate_category_scores_by_notice_type(HS_step7_df)
    HS_step9_df.to_csv(f"05_학사공지_카테고리분류_결과.csv", index=False, encoding='utf-8-sig') # <--------- 초기 데이터와, 저장 데이터 이름 / 저장 위치가 동일해야함
    print(f"  ▷ 완료 : 학사공지(카테고리)")
except Exception as e:
    print("  ▷ 실패")
    print(f"에러 내용: {e}")
#====================================================================================#




■ step1 시작
  ▷ 일반공지 완료: 4건
  ▷ 장학공지 완료: 13건
  ▷ 학사공지 완료: 5건

■ step2 시작


  ▷ 본문 크롤링 진행: 100%|██████████| 17/17 [00:25<00:00,  1.48s/it]


     ㄴ 완료 : 통합공지_(본문내용)


  ▷ 본문 크롤링 진행: 100%|██████████| 5/5 [00:08<00:00,  1.61s/it]



     ㄴ 완료 : 학사공지_(본문내용)
■ step3 시작


100%|██████████| 17/17 [00:00<00:00, 3342.23it/s]


  ▷ 완료 :통합공지(이미지링크)
■ step4 시작


100%|██████████| 17/17 [00:00<00:00, 3470.42it/s]
<ipython-input-27-cc4b01ae0a22>:678: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_normal["등록일"] = pd.to_datetime(combined_normal["등록일"], errors="coerce")
<ipython-input-27-cc4b01ae0a22>:678: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_normal["등록일"] = pd.to_datetime(combined_normal["등록일"], errors="coerce")
<ipython-input-27-cc4b01ae0a22>:678: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_normal["등록일"] = pd.to_datetime(combined_normal["등록일"], errors="coerce")


  ▷ 완료 : 통합공지(이미지텍스트)
■ step5 시작
  ▷ 완료 : 통합공지(마감일)
■ step6 시작
  ▷ 완료 : 통합공지(컬럼정렬)
■ step7 시작
  ▷ 완료 : 통합공지(데이터병합)
  ▷ 완료 : 학사공지(데이터병합)
■ step8 시작
  ▷ 완료 : 통합공지(마감현황)
■ final 시작
  ▷ 완료 : 통합공지(카테고리)
  ▷ 완료 : 학사공지(카테고리)


## 참고: 이전 버전 코드


<!-- hermes:explanation -->
## 이전 버전 통합 코드
아래 셀은 현재 대표 흐름 이전에 사용한 통합 코드 기록입니다. 함수 구조나 저장 순서가 일부 다를 수 있으므로, 변경 이력을 비교하거나 보조 참고 자료로 볼 수 있습니다.

재사용 전에는 입력 파일명, 컬럼 구조, 패키지 설치 상태, OCR 실행 환경을 먼저 확인해야 합니다.


In [ ]:
#step4. OCR 기능을 위한 라이브러리 설치
#-----------------------------------------------
!pip install easyocr
!pip install torch torchvision torchaudio

!apt install -y poppler-utils
!pip install pillow matplotlib
#-----------------------------------------------
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from urllib.parse import urljoin
import re
from tqdm import tqdm
import easyocr
import matplotlib.pyplot as plt
import numpy as np
import cv2
from PIL import Image
from io import BytesIO
from dateutil.parser import parse
from typing import Optional
import calendar

def crawl_step1(Noti_title, T_limit=100, T_max_pages=1):
    if Noti_title == '일반공지':
        NOTI = '01'
    elif Noti_title == '학사공지':
        NOTI = '02'
    elif Noti_title == '장학공지':
        NOTI = '03'
    else:
        raise ValueError("공지 유형은 '일반공지', '학사공지', '장학공지' 중 하나여야 합니다.")

    def crawl_page(limit, offset):
        url = f"https://www.jj.ac.kr/jj/community/notice{NOTI}.do?mode=list&&articleLimit={limit}&article.offset={offset}"
        try:
            res = requests.get(url, timeout=5)
            res.encoding = 'utf-8'
            res.raise_for_status()
        except Exception as e:
            print(f"[!] 요청 실패 (offset={offset}) → {e}")
            return []

        soup = bs(res.text, "html.parser")
        box = soup.find("tbody")
        rows = []

        if not box:
            return rows

        for tr in box.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) < 2:
                continue

            no = tds[0].text.strip()
            content_td = tds[1]

            a_tag = content_td.select_one("div > div > a")
            title = ' '.join(a_tag.stripped_strings) if a_tag else None
            href = a_tag.get("href") if a_tag else None
            link = urljoin(f"https://www.jj.ac.kr/jj/community/notice{NOTI}.do", href) if href else None
            dept_span = content_td.select_one("div > div:nth-of-type(3) > span.b-writer")
            dept = dept_span.text.strip() if dept_span else None
            date_span = content_td.select_one("div > div:nth-of-type(3) > span.b-date")
            date = date_span.text.strip() if date_span else None
            hit_span = content_td.select_one("div > div:nth-of-type(3) > span.b-hit")
            views = int(re.search(r'\d+', hit_span.text.strip()).group()) if hit_span else 0

            if no == "공지":
                if not dept:
                    dept = "관리자"
                date = datetime.today().strftime('%Y-%m-%d')

            prim_key = f"{no}+{dept}_{title[-20:]}" if title else None

            if title:
                rows.append({
                    "공지분류": Noti_title,
                    "등록 번호": no,
                    "기본 제목": prim_key,
                    "공지 제목": title,
                    "부서": dept,
                    "등록일": date,
                    "조회수": views,
                    "링크": link
                })

        return rows

    def gongji_crawl_parallel(limit=100, max_pages=80):
        offsets = [page * limit for page in range(max_pages)]
        all_rows = []

        with ThreadPoolExecutor(max_workers=10) as executor:
            results = list(executor.map(lambda offset: crawl_page(limit, offset), offsets))

        for page_rows in results:
            all_rows.extend(page_rows)

        df = pd.DataFrame(all_rows)
        df.drop_duplicates(subset=["공지 제목", "링크"], inplace=True)
        return df

    print(f"---> step1 시작: {Noti_title}")
    try:
        Tstep1_df = gongji_crawl_parallel(T_limit, T_max_pages)
        print(f"---> step1 완료: {Noti_title}, 총 {len(Tstep1_df)}건")
    except Exception as e:
        print("---> step1 실패")
        print(f"에러 내용: {e}")
        Tstep1_df = pd.DataFrame()

    return Tstep1_df

# 테스트 케이스 <--------------------- 실제 실행시에는 각 head()안에 숫자를 1에서 500으로 변경
df_general = crawl_step1("일반공지", 100, 5).head(500)
df_scholarship = crawl_step1("장학공지", 100, 5).head(500)
step1_df = pd.concat([df_general, df_scholarship], ignore_index=True)
HS_step1_df = crawl_step1("학사공지", 100, 5).head(500)

"""--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
#공지 내용 크롤링
# 📌 본문 크롤링 함수
def extract_notice_body(url):
    try:
        res = requests.get(url, timeout=5)
        res.encoding = 'utf-8'
        soup = bs(res.text, "html.parser")
        content_div = soup.select_one("div.b-content-box div.fr-view")
        if content_div:
            text = content_div.get_text(separator="", strip=True)  # \n 제거
            return text if text.strip() else None  # 빈 문자열이면 None 처리
        else:
            return None
    except Exception as e:
        print(f"[오류 발생] {url} → {e}")
        return None


def create_notice_content_df(df):
    temp_df = df.copy()
    contents = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        url = row["링크"]
        content = extract_notice_body(url)
        contents.append(content)

    temp_df["본문 내용"] = contents

    return temp_df

# step 2. 공지사항의 [본문내용 카테고리] 열 추가 ---> 시간 약간 소요 됨
print("---> step2 시작")
try:
    step2_df = create_notice_content_df(step1_df)
    HS_step2_df = create_notice_content_df(HS_step1_df)

    # 파일 저장
    step2_df.to_csv(f"통합공지_(본문내용까지).csv", index=False, encoding='utf-8-sig')
    print(f"---> step2 완료 : 저장파일명 = 통합공지_(본문내용까지)")
    HS_step2_df.to_csv(f"학사공지_(본문내용까지).csv", index=False, encoding='utf-8-sig')
    print(f"---> step2 완료 : 저장파일명 = 학사공지_(본문내용까지)")
except Exception as e:
    print("---> step2 실패")
    print(f"에러 내용: {e}")


"""------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
#공지사항에 있는 이미지 파일 링크 추출
# 🔹 이미지 링크 추출 함수
def extract_image_links(url):
    try:
        res = requests.get(url, timeout=5)
        res.encoding = 'utf-8'
        soup = bs(res.text, "html.parser")
        img_tags = soup.select("div.b-content-box div.fr-view img")

        # 🔹 이미지 링크 수집
        img_urls = []
        for img in img_tags:
            src = img.get("src")
            if src:
                # 절대경로가 아니면 urljoin으로 붙이기
                full_url = urljoin("https://www.jj.ac.kr", src)
                img_urls.append(full_url)

        # 🔹 리스트가 비어 있으면 None 반환
        if img_urls:
            # 🔸 문자열 형태로 반환 (리스트 [] 제거)
            return ", ".join(img_urls)
        else:
            return None
    except Exception as e:
        print(f"[이미지 크롤링 오류] {url} → {e}")
        return None

# 🔹 데이터프레임 병합 함수
def merge_image_links(df):
    # 이미지 링크 열 추가
    df["이미지 링크"] = None

    # 각 행에 대해 이미지 링크 추출
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        url = row["링크"]
        img_links = extract_image_links(url)
        df.at[idx, "이미지 링크"] = img_links

    return df

# step3. 공지사항에 있는 이미지 파일 링크를 추출하여 "이미지 링크"열 추가
# 약 10분 소요
print("---> step3 시작")
try:
    step3_df = merge_image_links(step2_df)
    # 파일 저장
    step3_df.to_csv(f"통합공지_(이미지링크까지).csv", index=False, encoding='utf-8-sig')
    print(f"---> step3 완료 : 저장파일명 = 통합공지_(이미지링크까지)")
except Exception as e:
    print("---> step3 실패")
    print(f"에러 내용: {e}")

"""---------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
# Step4. 이미지 파일의 링크를 읽어와서 텍스트 인식 수행
# 시간이 매우 많이 소요되므로 서버에서 실행 권장

# 따로 실행 시킬때는 아래 주석 제거 후 step4의 전체 코드 복사해서 실행

# 🔹 OCR 모델 초기화
reader = easyocr.Reader(['ko', 'en'])

# 🔹 이미지 링크에서 OCR 수행 함수
def ocr_image_links(img_urls):
    if not img_urls or pd.isna(img_urls):
        return None  # 이미지가 없는 경우

    # 이미지 링크가 여러 개일 수 있으므로 리스트로 변환
    img_list = img_urls.split(", ") if isinstance(img_urls, str) else [img_urls]

    # 🔹 각 이미지에 대해 OCR 수행
    texts = []
    for img_url in img_list:
        try:
            # 이미지 다운로드
            response = requests.get(img_url.strip())
            img = Image.open(BytesIO(response.content))
            img_np = np.array(img)

            # 🔹 OCR 수행
            result = reader.readtext(img_np, detail=0)
            texts.append(" ".join(result))
        except Exception as e:
            print(f"[이미지 OCR 오류] {img_url} → {e}")

    # 🔹 여러 이미지의 텍스트를 합쳐서 반환
    return " ".join(texts) if texts else None

# 🔹 데이터프레임 병합 함수
def df_to_ocr_image_links(df):

    # 각 행에 대해 이미지 링크 추출
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        img_urls = row.get("이미지 링크", "")
        img_texts = ocr_image_links(img_urls)
        df.at[idx, "이미지 텍스트"] = img_texts

    return df

# Step4. 시작>> 이미지 파일의 링크를 읽어와서 텍스트 인식 수행
print("---> step4 시작")
try:
    # 🔹 데이터프레임에 이미지 텍스트 병합 (시간 소요 가능)
    step4_df = df_to_ocr_image_links(step3_df)

    step4_df.to_csv(f"통합공지_(이미지텍스트까지).csv", index=False, encoding='utf-8-sig')
    print(f"---> step4 완료 : 저장파일명 = 통합공지_(이미지텍스트까지)")
except Exception as e:
    print("---> step4 실패")
    print(f"에러 내용: {e}")

"""-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
#step5. (현재 수정 중인 코드) -- 공지 내용에서 마감일 추출하여,[마감일자, 모집현황] 추가

import pandas as pd
import re
from datetime import datetime
from dateutil.parser import parse
from typing import Optional
import calendar

def preprocess_text(text: str) -> str:
    """지능형 전처리: 핵심 정보 보존, 노이즈 제거"""
    # 이메일/전화번호 제거 (날짜 패턴 보호)
    text = re.sub(r'\S+@\S+', '', text)  # 이메일
    text = re.sub(r'\d{2,4}-\d{3,4}-\d{4}', '', text)  # 전화번호
    # IR52 같은 코드명은 날짜 형식이 아니면 제거 (숫자-문자 조합, 단일 코드명)
    text = re.sub(r'\b(?!\d+[./]\d+)[A-Za-z]+\d+\b', '', text)
    # 요일 괄호 제거 (수), (금)
    text = re.sub(r'\([^)]*\)', '', text)
    # 날짜 형식 정규화 (2025. 4. 4 → 2025.4.4)
    text = re.sub(r'(\d+)\.\s+(\d+)', r'\1.\2', text)
    # 공백 정규화
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_until_deadline(text: str, base_year: int) -> Optional[str]:
    def build_date(year, month, day):
        try:
            if year < 100: year += 2000  # 2자리 연도 처리
            if 2000 <= year <= 2100 and 1 <= month <= 12 and 1 <= day <= 31:
                return f"{year}-{month:02d}-{day:02d}"
        except: pass
        return None

    text = re.sub(r'\([^)]*\)', '', text)  # 요일 제거

    # 패턴 그룹: (정규식, 그룹 인덱스, 후처리함수)
    patterns = [
        # 1. ~까지 신청/접수/마감
        (r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})일?\s*까지\s*(신청|접수|마감)', lambda m: build_date(int(m[1]), int(m[2]), int(m[3]))),

        # 2. YYYY년 MM월 DD일 ~ DD일 학생 모집
        (r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})[일\s]*[~\-]+\s*(\d{1,2})[일]?[^\n\r가-힣0-9]*.*?학생\s*모집', lambda m: build_date(int(m[1]), int(m[2]), int(m[4]))),

        # 3. 모집/신청/접수 기간
        (r'(서류\s*접수|접수\s*기간|신청\s*기간|모집\s*기간|모집기간|접수기간|신청기간|지원서\s*접수)[\s:：-]*'
         r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})일?\s*~\s*'
         r'(\d{4})?[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})일?',
         lambda m: build_date(int(m[5]) if m[5] else int(m[2]), int(m[6]), int(m[7]))),

        # 4. 마감 키워드 + 날짜
        (r'(마감|접수종료|신청마감)[\s:：]+'
         r'(\d{4})?[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})일?',
         lambda m: build_date(int(m[2]) if m[2] else base_year, int(m[3]), int(m[4]))),

        # 5. 상시모집 ~ 연도.월
        (r'상시모집.*~\s*(\d{4})[./년\s]*(\d{1,2})[월]?',
         lambda m: build_date(int(m[1]), int(m[2]), calendar.monthrange(int(m[1]), int(m[2]))[1])),

        # 6-1: YYYY.MM.DD HH:MM ~ MM.DD HH:MM
        (r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2}).*?~\s*'
         r'(\d{1,2})[./월\s]*(\d{1,2})',
         lambda m: build_date(int(m[1]), int(m[4]), int(m[5]))),

        # 6-2: ~YYYY.MM.DD HH:MM까지
        (r'~\s*(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2}).*?(?:까지|시)',
         lambda m: build_date(int(m[1]), int(m[2]), int(m[3]))),

        # 6-3: YYYY.MM.DD HH:MM ~ YYYY.MM.DD HH:MM
        (r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2}).*?~\s*'
         r'(\d{4})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})',
         lambda m: build_date(int(m[4]), int(m[5]), int(m[6]))),

        # 6-4: 2자리 연도 ~ 2자리 연도
        (r'(\d{2})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})\s*[~\-]\s*'
         r'(\d{2})[./년\s]*(\d{1,2})[./월\s]*(\d{1,2})',
         lambda m: build_date(2000 + int(m[4]), int(m[5]), int(m[6])))
    ]

    for pattern, extractor in patterns:
        if (match := re.search(pattern, text, re.IGNORECASE)):
            result = extractor(match)
            if result: return result

    return None


def enhanced_extractor(text: str, base_year: int) -> str:
    """최종 추출 엔진 (우선순위 개선)"""
    cleaned_text = preprocess_text(text)


    if deadline := extract_until_deadline(cleaned_text, base_year):
        return deadline


    # 안전하게 '-' 반환
    return '-'


def add_deadline_date(df: pd.DataFrame) -> pd.DataFrame:
    """최종 마감일자 추가 함수 (본문 우선 처리)"""
    def get_base_year(reg_date: str) -> int:
        try:
            return parse(reg_date, fuzzy=True).year
        except:
            return datetime.now().year

    def process_row(row):
      base_year = get_base_year(str(row.get('등록일', '')))

      # 1. 본문 내용 우선 처리
      main_text = str(row.get('본문 내용', ''))
      deadline = enhanced_extractor(main_text, base_year)
      if deadline != '-':
          return deadline

      # 2. 이미지 텍스트 차선 처리
      image_text = str(row.get('이미지 텍스트', ''))
      deadline = enhanced_extractor(image_text, base_year)
      if deadline != '-':
          return deadline

      # 3. 전체 텍스트 폴백 처리
      combined = ' '.join([
          str(row.get('공지 제목', '')),
          main_text,
          image_text
      ])
      deadline = enhanced_extractor(combined, base_year)
      return deadline

    def check_priority(row):
      combined_text = ' '.join([
          str(row.get('공지 제목', '')),
          str(row.get('본문 내용', '')),
          str(row.get('이미지 텍스트', ''))
      ])

      # 1. 선착순 키워드 우선
      if '선착순' in combined_text:
          return '선착순'

      # 2. 마감일 기준 모집중/마감 판단
      deadline = str(row.get('마감일자', '')).strip()
      try:
          if deadline and deadline != '-':
              deadline_date = datetime.strptime(deadline, "%Y-%m-%d").date()
              today = datetime.today().date()
              if deadline_date >= today:
                  return '모집중'
              else:
                  return '마감'
          else:
              return '정보 없음'
      except:
          return '정보 없음'


    df['마감일자'] = df.apply(process_row, axis=1)
    df['모집 현황'] = df.apply(check_priority, axis=1)
    return df

#step5. 시작) 공지 내용에서 마감일 추출하여,[마감일자, 모집현황] 추가
print("---> step5 시작")
try:
    # 🔹 마감일 추출 및 열 추가 (예: "모집 마감일" 등)
    step5_df = add_deadline_date(step4_df)
    HS_step5_df = add_deadline_date(HS_step2_df)

    #파일 저장
    step5_df.to_csv(f"통합공지_(마감일추출포함).csv", index=False, encoding='utf-8-sig')
    print(f"---> step5 완료 : 저장파일명 = 통합공지_(마감일추출포함)")
    HS_step5_df.to_csv(f"학사공지_(마감일추출포함).csv", index=False, encoding='utf-8-sig')
    print(f"---> 학사step5 완료 : 저장파일명 = 학사공지_(마감일추출포함)")
except Exception as e:
    print("---> step5 실패")
    print(f"에러 내용: {e}")



"""-------------------------------------------------------------------------------------------------------------------------------------------------------------------"""
#공비별 카테고리 키워드
일반_공지 = {
    "학사/수업": [
        "수강", "학기", "학점", "교육과정", "졸업", "교과목", "학사", "성적", "교양", "졸업생", "방학",
        "편입", "전공", "교환학생", "학업", "학위", "학력", "복학", "기숙사 입주",
        "학부", "학과", "수업", "실습", "학사", "수강신청", "교육"],
    "진로/취업": [
        "취업", "진로", "채용", "입사", "직무", "일자리", "인턴", "커리어", "직업", "잡콘서트",
        "일자리", "일자리박람회", "커리어"],
    "장학금/지원금": [
        "장학금", "장학", "장학생", "지원금", "학자금지원", "학자금 지원", "근로장학",  "근로 장학", "성적 장학", "성적장학", "등록금 감면", "장학 재단",
        "장학재단", "학비 지원", "학비지원"],
    "대외활동/서포터즈": [
        "서포터즈", "기자단", "홍보대사", "대외활동", "봉사단", "체험단", "멘토단", "자원봉사",
        "자원봉사자", "활동", "캠프", "행사", "봉사활동", "봉사자", "대사", "봉사", "응원단", "자원봉사", "자치기구"],
    "공모전/경진대회": [
        "공모전", "대회", "경진대회", "콘테스트", "아이디어", "캠프", "경진 대회", "해커톤"],
    "교내행사/이벤트": [
        "행사", "축제", "페스티벌", "박람회", "설명회", "콘서트", "캠프", "포럼", "세미나", "이벤트",
        "연합", "학회", "학술제", "학술대회", "워크숍", "온스타", "onSTAR", "초청", "참가자", "기념",
        "설명", "진행", "공연", "심포지엄", "전시", "라이브", "공모전", "경진대회", "문화제", "공연", "음악회",
        "단체 활동", "대동제", "대회", "페어", "파티", "축하", "컨퍼런스"],
    "비교과/교육": [
        "프로그램", "프로그램 안내", "프로그램안내", "지원 프로그램", "특강", "비교과", "워크숍", "세미나", "교육", "학습법",
        "연수", "강좌", "동아리", "소모임", "영어", "스터디", "멘토링", "온스타", "onSTAR", "예치금","역량",
        "과정", "실습", "학습", "역량 강화", "교육과정", "강의", "자격증", "취득", "학습자", "훈련", "강좌", "SP", "CP",
        "자격", "성장", "훈련", "학습법", "기초", "심화", "학점", "교양", "전공", "학습법", "성적", "공부", "실습", "과제"],
    "군 관련": [
        "예비군", "병역", "군입대", "훈련", "복무", "군대", "입대", "전역", "병사", "장교", "부사관", "훈련소",
        "군사", "복무기간", "병역의무", "군생활", "군 입대", "국방", "훈련소", "기초군사훈련", "동원훈련",
        "병역법", "군인", "군복무", "병역제도", "국방부", "예비군 훈련"],
    "창업 관련": [
          "창업", "창업 캠프"],
    "행정" : ["행정", "결과 안내"],
    "기타" : ["%%%%%%%"]
}

장학_공지 = {
    "국가근로장학금": [
  "국가근로장학금", "하계방학집중근로", "동계방학집중근로", "희망근로지 신청", "추가 신청", "1차 신청", "2차 신청", "3차 신청", "신규장학생 운영", "선발 발표", "사전교육", "근로 기간", "근로 장소", "교내근로", "교외근로", "공공기관", "시급", "최대 근로시간", "업무스케쥴", "출근부", "학업시간표 등록", "오리엔테이션", "부정근로", "근로자격 해제", "중복참여 불가", "장학생 준수사항", "근로지 배정", "선호학과", "근로지 담당자", "출근부 어플", "장학금 지급일", "근로지 대학제출", "학적변동", "휴학", "자퇴", "졸업", "제적", "문의", "시스템 오류", "한국장학재단 홈페이지", "모바일 어플"
],
"교내장학금": [
  "가족장학금", "수퍼스타(동문) 자녀 장학금", "다문화장학금", "장애인자녀장학금", "장애대학생장학금", "특수환경지원장학금", "장학사정관제장학금", "재단우대장학금", "신청기간", "신청자격", "성적기준", "이수학점", "평점평균", "제출서류", "가족관계증명서", "개인정보동의서", "장학금액", "지급방법", "inSTAR", "포털", "신규자", "중복수혜", "수업료 감면", "문의", "학생지원실"
],
"교외장학금": [
  "화성시인재육성재단", "하이트진로홀딩스", "강원랜드 멘토링", "전주시 글로벌 인재양성", "인천인재평생교육진흥원", "서울장학재단", "논산시장학회", "농어촌희망재단", "푸른등대 기부장학금", "부산진구장학회", "대산농촌재단", "고속도로장학재단", "은평구민장학재단", "춘천시민장학재단", "손태희장학재단", "협성문화재단", "롯데장학재단", "우아한 사장님 자녀", "미래산업인재", "지역정착 장학금", "해외연수", "해외봉사", "해외유학", "창업기숙사", "꿈드림", "멘토링", "사회리더", "기부장학", "본인부담 등록금 반값지원", "저소득 대학생", "독립유공자 후손", "북한이탈청소년", "소아청소년 당뇨인", "긴급재난 지원", "멘토단", "멘티", "활동기관", "활동계획서", "시급", "최대근로시간", "활동기간", "제출서류", "재단 홈페이지", "추천서"
],
"국가장학금": [
  "국가장학금", "이공계", "인문100년", "예술체육비전", "주거안정장학금", "신청기간", "1차", "2차", "가구원 동의", "소득분위", "지원구간", "성적기준", "신청방법", "서류제출", "지급", "선발", "학업계획서", "학업시간표", "학적변동", "구제신청", "탈락", "문의", "한국장학재단"
],
"학자금대출": [
  "학자금대출", "대출이자 지원", "신용회복 지원", "특별상환유예", "기등록자 특별승인", "농촌출신 학자금융자", "채무자 신고", "해외이주", "유학신고", "업무처리기준", "신청기간", "신청방법", "심사", "지급", "문의", "상환", "학자금지원사업", "결혼이민자", "혼인귀화자", "대학학비지원", "지방자치단체", "농어촌", "이자지원사업", "상환유예", "신용회복"
],
"멘토링/청소년교육지원": [
  "대학생 청소년교육지원사업", "멘토링", "사회리더", "멘토", "멘티", "활동계획서", "활동기관", "사전신청", "사전교육", "활동기간", "활동내용", "시급", "최대근로시간", "중복참여", "부정근로", "선발", "결과발표", "지급", "지급일", "지급방법", "온라인교육", "매뉴얼", "시스템", "어플", "홈페이지", "활동 포기", "기관-학생 상호평가", "출근부", "학업시간표 입력", "업무스케쥴", "활동기관 발굴", "신규기관 등록", "중복참여불가", "이해관계 회피", "활동시간 제한"
],
"특별지원/목회자/북한이탈/보훈": [
  "목회자 장학금", "목회자 자녀 장학금", "북한이탈주민 교육지원금", "보훈장학금", "국가보훈대상자", "국가유공자", "교육지원대상자증명서", "대학수업료등면제대상자증명서", "가족관계증명서", "재직증명서", "교회주보", "결혼이민자", "혼인귀화자", "다문화", "장애인자녀", "장애대학생", "특수환경지원", "아동복지시설", "한부모가정", "긴급재난 지원", "제출서류", "신청기간", "지급방법"
],
"글로벌/해외연수/파란사다리": [
  "글로벌 인재양성", "영어능력강화", "해외연수", "어학연수", "교환학생", "파란사다리", "Fulbright", "국제교류", "신청기간", "선발", "지원", "서류제출", "학점인정", "학적변동", "재단 홈페이지"
],
"취업연계/창업/고졸후학습자/희망사다리": [
  "중소기업 취업연계 장학금", "희망사다리", "고졸후학습자", "창업", "취업", "취업지원", "창업지원", "직무기초교육", "의무종사", "취업연계", "창업지원금", "의무재직", "일자리사업 중복참여", "사용계획서", "진단평가", "보증보험", "취업연계 상담센터", "지원사업", "심사기준", "지원금", "등록금 전액지원", "의무종사 기업기준", "재직확인"
],
"행정/안내/기타": [
  "통학로", "학생지원", "정보로", "안내", "제한대학", "지원대상", "지원방법", "제출서류", "문의", "본인부담", "등록금", "반값지원", "지역정착", "졸업", "휴학", "자퇴", "제적", "졸업예정자", "졸업유예", "학점", "평점", "성적", "학업", "학적변동", "inSTAR", "포털", "이메일", "방문접수", "팩스", "제출기한", "신청기한", "지급기한", "시스템오류", "어플오류", "홈페이지오류", "콜센터", "공지사항", "개인정보동의", "계좌등록", "지급계좌", "통장", "담당자", "연락처"
]
}
학사_공지 = {
    "수업/강의 관련": ["강의", "수업", "교과목", "강의실", "강좌", "시간표", "수강신청", "보강"],
    "시험/평가": ["시험", "수시고사", "중간고사", "기말고사", "평가", "성적", "과제", "응시"],
    "졸업/학위": ["졸업", "논문", "학위", "심사", "학점", "수료", "취업", "조기"],
    "등록/휴학/복학": ["등록", "휴학", "복학", "신청", "복학생", "편입", "자퇴"],
    "장학/재정": ["장학금", "지원", "신청서", "선발", "지급", "장학생"],
    "행사/일정": ["설명회", "일정", "행사", "참석", "기념일", "주간", "날짜", "개강", "종강", "연휴"],
    "행정/공지": ["공지", "안내", "서류", "제출", "처리", "관리", "참조", "변경"]
}
부서_분류 = {
    "학사/수업": ["학생 지원", "학생지원", "학생 역량", "학생역량", "학사지원", "학사 지원", "LINC"],
    "진로/취업":  ["대학 일자리", "대학일자리", "진로취업", "진로 취업", "학생 성공", "학생성공"],
    "장학금/지원금": [],
    "대외활동/서포터즈": [],
    "공모전/경진대회":[],
    "교내행사/이벤트": ["대학 일자리", "대학일자리", "진로취업", "진로 취업" "학생 성공", "학생성공"],
    "비교과/교육": ["대학 일자리", "대학일자리", "진로취업", "진로 취업" "학생 성공", "학생성공"],
    "군 관련": ["예비군"],
}
"""-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"""


# 🔹 키워드 정의 (간략 버전)
category_dict = {
    "일반공지": 일반_공지,
    "장학공지": 장학_공지
}
department_dict = {
    "일반공지": 부서_분류,
    "장학공지": 부서_분류
}
# 가중치 설정
TITLE_WEIGHT = 3
CONTENT_WEIGHT = 2
DEPT_WEIGHT = 0
IMAGE_WEIGHT = 2

def calculate_category_scores_by_notice_type(df: pd.DataFrame) -> pd.DataFrame:

    def calculate_score(title, content, dept, image, category_keywords, department_keywords):
        title = str(title) if pd.notna(title) else ""
        content = str(content) if pd.notna(content) else ""
        dept = str(dept) if pd.notna(dept) else ""
        image = str(image) if pd.notna(image) else ""

        scores = {k: 0 for k in category_keywords.keys()}

        for cat, keywords in category_keywords.items():
            scores[cat] += (sum(title.count(k) for k in keywords) * TITLE_WEIGHT * 10) / (len(title) + 1)
            scores[cat] += (sum(content.count(k) for k in keywords) * CONTENT_WEIGHT * 10) / (len(content) + 1)
            scores[cat] += (sum(image.count(k) for k in keywords) * IMAGE_WEIGHT * 10) / (len(image) + 1)

        for cat, keywords in department_keywords.items():
            if cat in scores:
                scores[cat] += (sum(dept.count(k) for k in keywords) * DEPT_WEIGHT * 10) / (len(dept) + 1)

        if all(v == 0 for v in scores.values()):
            scores["기타"] = 1.0

        return scores

    # 점수 계산 및 열 추가
    result_df = df.copy()
    for idx, row in result_df.iterrows():
        noti_type = row.get("공지분류", "").strip()
        category_keywords = category_dict.get(noti_type, {})
        department_keywords = department_dict.get(noti_type, {})

        title = row.get("공지 제목", "")
        content = row.get("본문 내용", "")
        dept = row.get("부서", "")
        image = row.get("이미지 텍스트", "")

        scores = calculate_score(title, content, dept, image, category_keywords, department_keywords)

        for cat, score in scores.items():
            colname = f"{noti_type}_{cat}"
            if colname not in result_df.columns:
                result_df[colname] = 0.0
            result_df.at[idx, colname] = score

    return result_df

# 필수 컬럼 리스트 (순서 보장)
required_columns = [
    '공지분류', '등록 번호', '기본 제목', '공지 제목',
    '부서', '등록일', '조회수', '링크', '본문 내용',
    '이미지 링크', '이미지 텍스트', '마감일자', '모집 현황'
]

# 누락된 컬럼 추가
def index_test(df: pd.DataFrame) -> pd.DataFrame:
  for col in required_columns:
      if col not in df.columns:
          # 데이터 타입에 맞게 초기화
          if col == '조회수':
              df[col] = 0  # 숫자형
          elif col in ['등록일', '마감일자']:
              df[col] = pd.NaT  # 날짜형
          else:
              df[col] = ''  # 문자열
  return df
df6 = index_test(step5_2_df)
# 컬럼 순서 재정렬
df6 = df6.reindex(columns=required_columns)
df6.to_csv("test6.csv", index=False, encoding='utf-8-sig')

#step6. 카테고리분류 지정
print("---> step6 시작")
try:
    step6_df = calculate_category_scores_by_notice_type(df6)

# 🔹 키워드 정의 (간략 버전)
category_dict = {
    "일반공지": 일반_공지,
    "장학공지": 장학_공지
}
department_dict = {
    "일반공지": 부서_분류,
    "장학공지": 부서_분류
}
# 가중치 설정
TITLE_WEIGHT = 3
CONTENT_WEIGHT = 2
DEPT_WEIGHT = 0
IMAGE_WEIGHT = 2

def calculate_category_scores_by_notice_type(df: pd.DataFrame) -> pd.DataFrame:

    def calculate_score(title, content, dept, image, category_keywords, department_keywords):
        title = str(title) if pd.notna(title) else ""
        content = str(content) if pd.notna(content) else ""
        dept = str(dept) if pd.notna(dept) else ""
        image = str(image) if pd.notna(image) else ""

        scores = {k: 0 for k in category_keywords.keys()}

        for cat, keywords in category_keywords.items():
            scores[cat] += (sum(title.count(k) for k in keywords) * TITLE_WEIGHT * 10) / (len(title) + 1)
            scores[cat] += (sum(content.count(k) for k in keywords) * CONTENT_WEIGHT * 10) / (len(content) + 1)
            scores[cat] += (sum(image.count(k) for k in keywords) * IMAGE_WEIGHT * 10) / (len(image) + 1)

        for cat, keywords in department_keywords.items():
            if cat in scores:
                scores[cat] += (sum(dept.count(k) for k in keywords) * DEPT_WEIGHT * 10) / (len(dept) + 1)

        if all(v == 0 for v in scores.values()):
            scores["기타"] = 1.0

        return scores

    # 점수 계산 및 열 추가
    result_df = df.copy()
    for idx, row in result_df.iterrows():
        noti_type = row.get("공지분류", "").strip()
        category_keywords = category_dict.get(noti_type, {})
        department_keywords = department_dict.get(noti_type, {})

        title = row.get("공지 제목", "")
        content = row.get("본문 내용", "")
        dept = row.get("부서", "")
        image = row.get("이미지 텍스트", "")

        scores = calculate_score(title, content, dept, image, category_keywords, department_keywords)

        for cat, score in scores.items():
            colname = f"{noti_type}_{cat}"
            if colname not in result_df.columns:
                result_df[colname] = 0.0
            result_df.at[idx, colname] = score

    return result_df

"""------------------------------------------------------------------------------------------------------------------------------------------------"""
#step6. 카테고리분류 지정
print("---> step6 시작")
try:
    step6_df = calculate_category_scores_by_notice_type(step5_df)
    HS_step6_df = calculate_category_scores_by_notice_type(HS_step5_df)

    #파일 저장
    step6_df.to_csv(f"04_통합공지_카테고리분류_결과.csv", index=False, encoding='utf-8-sig')
    print(f"---> step6 완료 : 저장파일명 = 04_통합공지_카테고리분류_결과")
    HS_step6_df.to_csv(f"05_학사공지_카테고리분류_결과.csv", index=False, encoding='utf-8-sig')
    print(f"---> step6 완료 : 저장파일명 = 05_학사공지_카테고리분류_결과")
except Exception as e:
    print("---> step6 실패")
    print(f"에러 내용: {e}")


# 🔹 최종 데이터프레임 출력
step6_df


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 664.8/664.8 MB 69.1 MB/s eta 0:00:01
ERROR: Operation cancelled by user
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)


ModuleNotFoundError: No module named 'easyocr'